# Notebook 4: Rolling Estimation & Regime Analysis
**Project:** GARCH & Volatility Forecasting
**Author:** Niraj Neupane | github.com/nirajneupane17

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from arch import arch_model
import statsmodels.api as sm
from scipy import stats
import sys; sys.path.insert(0, '../src')
import warnings; warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize':(13,6),'font.size':11,'axes.titlesize':13,'axes.titleweight':'bold'})
returns = pd.read_csv('../data/returns.csv', index_col='Date', parse_dates=True)
spy = returns['SPY'] * 100
spy_raw = returns['SPY']
print(f'Loaded {len(spy):,} obs | {returns.index[0].date()} to {returns.index[-1].date()}')


Loaded 2,609 obs | 2015-01-01 to 2024-12-31


In [2]:
from rolling_vol import rolling_volatility, classify_regimes, regime_statistics
rv_df = rolling_volatility(spy_raw, windows=[21,63,126,252])
print(rv_df.tail(5).round(4))

            21d_vol  63d_vol  126d_vol  252d_vol
Date                                            
2024-12-25   0.1387   0.1595    0.1543    0.1571
2024-12-26   0.1434   0.1610    0.1550    0.1573
2024-12-27   0.1455   0.1614    0.1548    0.1561
2024-12-30   0.1486   0.1602    0.1552    0.1561
2024-12-31   0.1448   0.1598    0.1542    0.1564


In [3]:
fig,ax=plt.subplots(figsize=(13,6))
colors=['#185FA5','#E24B4A','#1D9E75','#7B1FA2']
for col,c in zip(rv_df.columns,colors):
    ax.plot(rv_df.index,rv_df[col]*100,color=c,linewidth=1.2,label=col,alpha=0.85)
ax.set_title('Rolling Annualised Volatility — Multiple Windows')
ax.set_ylabel('Annualised Volatility (%)'); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/05_rolling_volatility_windows.png',dpi=150,bbox_inches='tight')
plt.show()

In [4]:
vol_63=rv_df['63d_vol'].dropna()
regimes=classify_regimes(vol_63)
stats_df=regime_statistics(spy_raw,vol_63)
print('Regime Statistics:')
print(stats_df.round(4))
regimes.value_counts().to_csv('../results/volatility_regimes.csv',header=['count'])

Regime Statistics:
        count  mean_return  annual_vol  sharpe
regime                                        
Low       841       0.0084      0.1479   0.057
Medium    865      -0.0145      0.1613  -0.090
High      841      -0.1387      0.2104  -0.659


In [5]:
import matplotlib.patches as mpatches
colors_r={'Low':'#1D9E75','Medium':'#F57C00','High':'#E24B4A'}
vol_pct=vol_63*100
fig,axes=plt.subplots(2,1,figsize=(13,9))
ax1=axes[0]
low_t=vol_pct.quantile(0.33); high_t=vol_pct.quantile(0.67)
for date,v in vol_pct.items():
    r='Low' if v<low_t else ('Medium' if v<high_t else 'High')
    ax1.bar(date,v,color=colors_r[r],alpha=0.65,width=1)
ax1.axhline(low_t,color='#1D9E75',linewidth=1.3,linestyle='--')
ax1.axhline(high_t,color='#E24B4A',linewidth=1.3,linestyle='--')
patches=[mpatches.Patch(color=c,label=l) for l,c in colors_r.items()]
ax1.legend(handles=patches,fontsize=9)
ax1.set_title('Volatility Regime Classification (63-Day Rolling)')
ax2=axes[1]
ewma_v=spy_raw.ewm(span=63).std()*np.sqrt(252)*100
ax2.plot(vol_pct.index,vol_pct,color='#185FA5',linewidth=1.2,label='63d Rolling')
ax2.plot(ewma_v.index,ewma_v,color='#E24B4A',linewidth=1.2,linestyle='--',label='63d EWMA')
ax2.set_title('Rolling vs EWMA Volatility'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.savefig('../results/06_volatility_regimes.png',dpi=150,bbox_inches='tight')
plt.show()